# SEISMICPIPELINE: PREDICCION DE RIESGO DE TSUNAMI MEDIANTE XGBOOST

<br>

**Institucion:** Universidad Internacional del Ecuador (UIDE)

**Escuela:** Ciencias de la Computacion

**Asignaturas:** Big Data - Machine Learning - Gestion de Proyectos de SI - Ciberseguridad

**Semestre:** Sexto - 2026

**Jira:** Proyecto SEIS

**Repositorio:** github.com/DanielSozoranga/tsunami-risk-pipeline

<br>

**Equipo Scrum:**

| Integrante | Roles |
|---|---|
| **Daniel Sozoranga** | Scrum Master · Development Team |
| **Ricardo Álvarez** | Product Owner · Development Team |

<br>

---

## PREGUNTA QUE RESPONDE EL MODELO

**?Este sismo generara tsunami?**

El modelo es un clasificador binario **XGBoost** que, dadas las caracteristicas fisicas de un sismo, devuelve una **probabilidad continua entre 0 y 1** (`predict_proba`) de que el evento genere tsunami. Esa probabilidad se proyecta sobre el registro sismico costero ecuatoriano como un **score de riesgo por provincia** (Esmeraldas, Manabi, Santa Elena, Guayas, El Oro, Galapagos), no como clasificacion binaria.

<br>

---

## ESTRUCTURA DEL NOTEBOOK

| Seccion | Contenido |
|---|---|
| **Seccion 1** | Instalacion y Configuracion del Entorno |
| **Seccion 2** | Importacion de Librerias |
| **Seccion 3** | Conexion y Extraccion de Datos (API USGS) |
| **Seccion 4** | Variables Obtenidas - Dataset Original |
| **Seccion 5** | Renombrado de Variables: API -> Espanol |
| **Seccion 6** | Limpieza y Preprocesamiento (ETL) |
| **Seccion 7** | Analisis Exploratorio (EDA) |
| **Seccion 8** | Benchmarking de Engines: Pandas vs PySpark vs Dask |
| **Seccion 9** | Preparacion del Dataset de Machine Learning |
| **Seccion 10** | Entrenamiento del Clasificador XGBoost |
| **Seccion 11** | Diagnostico de Entrenamiento |
| **Seccion 12** | Evaluacion del Modelo |
| **Seccion 13** | Modelos Baseline de Comparacion |
| **Seccion 14** | Justificacion Tecnica de la Seleccion de XGBoost |
| **Seccion 15** | Proyeccion de Riesgo sobre Ecuador |
| **Seccion 16** | Validacion Internacional (Tohoku 2011 / Maule 2010) |
| **Seccion 17** | Dashboard Interactivo de Resultados |
| **Seccion 18** | Analisis Critico: Limitaciones y Mejoras |
| **Seccion 19** | Conclusiones |

<br>

---

## CONVENCIONES

- **Semilla aleatoria global:** `SEED = 42` - reutilizada en todo split y modelo para garantizar reproducibilidad.
- **Idioma de las variables:** los datos llegan de la API con nombres en ingles. En la Seccion 5 se renombran a espanol con tabla de equivalencias original -> final.
- **Features del modelo (7):** `magnitud`, `profundidad_km`, `latitud`, `longitud`, `significancia`, `num_estaciones`, `brecha_azimutal`. La seleccion se justifica en la Seccion 7 con la matriz de correlacion completa.
- **Target:** `tsunami` (0 = no genero tsunami, 1 = si genero tsunami), bandera oficial del catalogo USGS.
- **Modelo unico:** XGBoost. Random Forest y Regresion Logistica aparecen solo como baselines de comparacion (Seccion 13).

<br>

---

# SECCION 1 - INSTALACION Y CONFIGURACION DEL ENTORNO

<br>

Se instalan las librerias que no estan disponibles por defecto en Google Colab.

| Libreria | Proposito |
|---|---|
| **xgboost** | Algoritmo principal del proyecto (clasificador de tsunamis) |
| **pyspark** | Procesamiento distribuido (motor Big Data del benchmarking) |
| **dask** | Procesamiento paralelo con ejecucion lazy (benchmarking) |
| **plotly** | Dashboard interactivo de resultados |
| **psutil** | Medicion de consumo de memoria RAM en tiempo real |
| **joblib** | Serializacion del modelo entrenado |

<br>

In [ ]:
# Librerias que Google Colab no tiene preinstaladas.
# Si ejecutas localmente, asegurate de tener Python 3.10+
# y un entorno virtual activo antes de correr este comando.
!pip install xgboost pyspark "dask[dataframe]" plotly psutil joblib --quiet

print("Librerias instaladas correctamente")


---

# SECCION 2 - IMPORTACION DE LIBRERIAS

<br>

In [ ]:
# --- Consumo de API y utilidades ---
import os
import time
import math
import json
import warnings
import requests

# --- Manipulacion de datos ---
import numpy  as np
import pandas as pd

# --- Visualizacion estatica ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Visualizacion interactiva ---
import plotly.graph_objects as go
import plotly.express       as px
from   plotly.subplots      import make_subplots

# --- Machine Learning ---
from sklearn.model_selection  import (train_test_split, StratifiedKFold,
                                      cross_val_score)
from sklearn.metrics          import (roc_auc_score, roc_curve,
                                      confusion_matrix, classification_report,
                                      precision_score, recall_score, f1_score,
                                      ConfusionMatrixDisplay)
from sklearn.ensemble         import RandomForestClassifier
from sklearn.linear_model     import LogisticRegression
from sklearn.pipeline         import Pipeline
from sklearn.preprocessing    import StandardScaler
from xgboost                  import XGBClassifier
import xgboost as xgb

# --- Persistencia y metricas del sistema ---
import joblib   # Serializacion del modelo entrenado
import psutil   # Medicion de consumo de RAM en tiempo real

# --- Configuracion global de visualizacion ---
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 5)

# Semilla global: garantiza reproducibilidad en splits, modelos y muestras
SEED = 42
np.random.seed(SEED)

print("Librerias importadas correctamente")
print(f"  pandas  : {pd.__version__}")
print(f"  numpy   : {np.__version__}")
print(f"  xgboost : {xgb.__version__}")
print(f"  SEED    : {SEED}")


---

# SECCION 3 - CONEXION Y EXTRACCION DE DATOS (API USGS)

<br>

## 3.1 Descripcion de la Fuente

<br>

| Parametro | Detalle |
|---|---|
| **Endpoint** | https://earthquake.usgs.gov/fdsnws/event/1/query |
| **Autenticacion** | Ninguna (API 100% publica) |
| **Estandar** | FDSN Event Web Service Specification |
| **Formato de respuesta** | GeoJSON (FeatureCollection) |
| **Cobertura temporal** | 1900 - presente |
| **Cobertura geografica** | Global |

<br>

## 3.2 Parametros de Consulta

<br>

Se extrae el **Cinturon de Fuego del Pacifico (1990-2024)** con los siguientes parametros:

| Parametro | Valor | Justificacion |
|---|---|---|
| `minmagnitude` | 5.0 | Eventos con potencial tsunamigenico real |
| `minlatitude` / `maxlatitude` | -60 / 65 | Desde la placa Antartica hasta Alaska/Kamchatka |
| `minlongitude` / `maxlongitude` | 110 / 300 | Cuenca del Pacifico (300 = -60W, permite cruzar el antimeridiano) |
| `limit` | 20000 | Limite maximo de la API por request -> requiere paginacion anual |

La paginacion anual (1990-2024 = 35 requests) evita superar el limite de la API. El retry con backoff exponencial protege contra fallas transitorias de la red.

<br>

## 3.3 Configuracion del Entorno de Archivos

<br>

In [ ]:
# =============================================================
#  SECCION 3.3 - CONFIGURACION DE CARPETAS
# =============================================================
#  Estructura estandar del proyecto:
#    /data/raw        <- datasets crudos directamente de la API
#    /data/processed  <- datasets limpios y artefactos del modelo
#    /notebooks       <- notebooks de analisis y exploracion
#    /models          <- modelos serializados (.pkl)
# =============================================================

try:
    # Entorno Google Colab: monta Drive automaticamente
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/SeismicPipeline'
except ModuleNotFoundError:
    # Entorno local: crea las carpetas relativas al directorio actual
    BASE = os.path.abspath('./SeismicPipeline')

DIRS = {
    'raw'       : f'{BASE}/data/raw',
    'processed' : f'{BASE}/data/processed',
    'notebooks' : f'{BASE}/notebooks',
    'models'    : f'{BASE}/models',
}

for nombre, ruta in DIRS.items():
    os.makedirs(ruta, exist_ok=True)   # exist_ok: no falla si ya existe
    print(f"  [OK] {nombre:10s} -> {ruta}")

print()
print("Estructura de carpetas configurada correctamente")


---

> **Nota de desarrollo:** Esta seccion corresponde al issue **SEIS-6** (estructura y entorno base).
> Las secciones 3.4 en adelante (extraccion API, ETL, modelado y evaluacion) se desarrollan
> en los commits subsiguientes: SEIS-7, SEIS-8, SEIS-9... hasta SEIS-28.
